In [1]:
import pandas as pd
import numpy as np

from linearmodels import PanelOLS

In [2]:
data = pd.read_csv('cleaned_school_data.csv')  

In [3]:
print(data.groupby("Year")["mean_scale_score"].agg(["mean", "std", "min", "max"]).round(3))

         mean     std    min    max
Year                               
2018  599.896  11.460  555.0  647.0
2019  599.042  11.667  553.0  646.0
2022  599.976  11.671  554.0  650.0


In [4]:
# adjust % poverty to a percentage scale
data['poverty_percentage'] = (data['% Poverty'] * 100).round(3)

In [5]:
# # preliminary DiD variables
# data['post'] = (data['Year'] == 2022).astype(int)
# data['treated_cont'] = data['post'] * data['poverty_percentage']

### Individual Grade Datasets

In [6]:
# grade splits

data_3 = data[
    (data['Grade'] == '3') & 
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]


data_4 = data[
    (data['Grade'] == '4') &  
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_5 = data[
    (data['Grade'] == '5') &
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_6 = data[
    (data['Grade'] == '6') &   
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_7 = data[
    (data['Grade'] == '7') &
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_8 = data[
    (data['Grade'] == '8') &
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]



### Verifying Parallel Trends Assumption in the Pre Period

#### All Grades

In [7]:
# Time-placebo test: use 2018 and 2019 as "pre" period
# this is a method to test the parallel trends assumption by checking for any pre-existing trends in the outcome variable before the treatment period
pre_data = data[data['Year'].isin([2018,2019])].copy() # new pre data
pre_data['fake_post'] = (pre_data['Year'] == 2019).astype(int) # treating 2019 as the "post" period in the placebo test
pre_data['fake_treated_cont'] = pre_data['fake_post'] * pre_data['poverty_percentage'] # new placebo interaction term for treated

In [8]:
pre_data = pre_data.set_index(['DBN', 'Year']) # set index for PanelOLS, it requires a multi-index with entity and time dimensions

In [9]:
# testing parallel trends with a placebo DiD model using PanelOLS
parallel_trends_model = PanelOLS(
    dependent=pre_data['mean_scale_score'],
    exog=pre_data[['fake_treated_cont']],
    entity_effects=True,
    time_effects=True,
    weights = pre_data['number_tested']
).fit(cov_type='clustered', cluster_entity=True)

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [10]:
print(f"Placebo test coefficient: {parallel_trends_model.params['fake_treated_cont']:.4f}, p-value: {parallel_trends_model.pvalues['fake_treated_cont']:.4f}")

Placebo test coefficient: -0.0016, p-value: 0.5982


```python
Placebo test coefficient: -0.0009, p-value: 0.8371
``` 

This is great!!!! Near zero trend in pre-period, parallel trends assumption met

#### Same thing for individual grades

In [11]:
pre_data3 = data_3[data_3['Year'].isin([2018,2019])].copy() # new pre data
pre_data3['fake_post'] = (pre_data3['Year'] == 2019).astype(int) # treating 2019 as the "post" period in the placebo test
pre_data3['fake_treated_cont'] = pre_data3['fake_post'] * pre_data3['poverty_percentage'] # new placebo interaction term for treated

pre_data3 = pre_data3.set_index(['DBN', 'Year'])


# testing parallel trends with a placebo DiD model using PanelOLS
parallel_trends_model3 = PanelOLS(
    dependent=pre_data3['mean_scale_score'],
    exog=pre_data3[['fake_treated_cont']],
    entity_effects=True,
    time_effects=True,
    weights = pre_data3['number_tested']
).fit(cov_type='clustered', cluster_entity=True)

print(f"Placebo test coefficient: {parallel_trends_model3.params['fake_treated_cont']:.4f}, p-value: {parallel_trends_model3.pvalues['fake_treated_cont']:.4f}")

Placebo test coefficient: 0.0230, p-value: 0.0182


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [12]:
results = {}

for i in range(3, 9):
    data = globals()[f"data_{i}"]
    
    pre_data = data[data['Year'].isin([2018, 2019])].copy()
    pre_data['fake_post'] = (pre_data['Year'] == 2019).astype(int)
    pre_data['fake_treated_cont'] = pre_data['fake_post'] * pre_data['poverty_percentage']
    
    pre_data = pre_data.set_index(['DBN', 'Year'])
    
    model = PanelOLS(
        dependent=pre_data['mean_scale_score'],
        exog=pre_data[['fake_treated_cont']],
        entity_effects=True,
        time_effects=True,
        weights=pre_data['number_tested']
    ).fit(cov_type='clustered', cluster_entity=True)
    
    results[i] = model
    
    print(f"Dataset {i} → Coef: {model.params['fake_treated_cont']:.4f}, "
          f"p-value: {model.pvalues['fake_treated_cont']:.4f}")

Dataset 3 → Coef: 0.0230, p-value: 0.0182
Dataset 4 → Coef: 0.0289, p-value: 0.0036


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Dataset 5 → Coef: -0.0058, p-value: 0.5272
Dataset 6 → Coef: -0.0395, p-value: 0.0011


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Dataset 7 → Coef: -0.0396, p-value: 0.0040
Dataset 8 → Coef: -0.0007, p-value: 0.9570


A very scary output to see when assessing parallel trends. Luckily, I am not the first person to pursue a project using DID and have assumptions fail. The difficulty with this model is the alleged binary of "yes" or "no" when talking about parallel trends violation. It is unrealistic to expect consistency when dealing woth real-world data - the world is imperfect and things tend to fluctuate. <br> 

The question is, how do you move forward from this? Rambachan & Roth (2023) have a different methodological approach involving a sensitivity analysis. Testing for parallel trends is already problematic to begin with, but still may have some insight into pre-trends. This paper suggests that instead of passing or failing the assumption to assess how large of a violation does it need to be to affect results. The researchers developed an HonestDID package, which allows for sensitivity analysis towards the magnitude of the assumption violation. <br>

This is where things get complicated. HonestDID is strictly an R and Stata package. Also, it only works with event studies. This may be a more advanced and insightful model. 

### Event Studies Model

In [13]:
grade_data = pd.read_csv('cleaned_school_data_updated.csv')

In [14]:
grade_data

,Report Category,DBN,school_name,Grade,Year,Student Category,number_tested,mean_scale_score,level_1_count,level_1_percentage,level_2_count,level_2_percentage,level_3_count,level_3_percentage,level_4_count,level_4_percentage,level_3_4_count,level_3_4_percentage,% Poverty,Poverty_Category
0,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2022,All Students,21,594.0,4.0,19.0,12.0,57.1,4.0,19.0,1.0,4.8,5.0,23.8,0.838,1
1,School,01M015,P.S. 015 ROBERTO CLEMENTE,4,2022,All Students,30,596.0,6.0,20.0,14.0,46.7,5.0,16.7,5.0,16.7,10.0,33.3,0.838,1
2,School,01M015,P.S. 015 ROBERTO CLEMENTE,5,2022,All Students,23,599.0,11.0,47.8,4.0,17.4,4.0,17.4,4.0,17.4,8.0,34.8,0.838,1
3,School,01M015,P.S. 015 ROBERTO CLEMENTE,All Grades,2022,All Students,74,596.0,21.0,28.4,30.0,40.5,13.0,17.6,10.0,13.5,23.0,31.1,0.838,1
4,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2019,All Students,27,606.0,1.0,3.7,7.0,25.9,18.0,66.7,1.0,3.7,19.0,70.4,0.845,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14462,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,All Grades,2019,All Students,323,593.0,113.0,35.0,114.0,35.3,73.0,22.6,23.0,7.1,96.0,29.7,0.949,1
14463,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,6,2018,All Students,111,595.0,37.0,33.3,37.0,33.3,26.0,23.4,11.0,9.9,37.0,33.3,0.915,1
14464,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,7,2018,All Students,116,592.0,50.0,43.1,48.0,41.4,18.0,15.5,0.0,0.0,18.0,15.5,0.915,1
14465,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,8,2018,All Students,90,591.0,24.0,26.7,48.0,53.3,16.0,17.8,2.0,2.2,18.0,20.0,0.915,1


In [15]:
grade_data['poverty_percentage'] = (grade_data['% Poverty'] * 100).round(3)

In [16]:
grade_data

,Report Category,DBN,school_name,Grade,Year,Student Category,number_tested,mean_scale_score,level_1_count,level_1_percentage,...,level_2_percentage,level_3_count,level_3_percentage,level_4_count,level_4_percentage,level_3_4_count,level_3_4_percentage,% Poverty,Poverty_Category,poverty_percentage
0,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2022,All Students,21,594.0,4.0,19.0,...,57.1,4.0,19.0,1.0,4.8,5.0,23.8,0.838,1,83.8
1,School,01M015,P.S. 015 ROBERTO CLEMENTE,4,2022,All Students,30,596.0,6.0,20.0,...,46.7,5.0,16.7,5.0,16.7,10.0,33.3,0.838,1,83.8
2,School,01M015,P.S. 015 ROBERTO CLEMENTE,5,2022,All Students,23,599.0,11.0,47.8,...,17.4,4.0,17.4,4.0,17.4,8.0,34.8,0.838,1,83.8
3,School,01M015,P.S. 015 ROBERTO CLEMENTE,All Grades,2022,All Students,74,596.0,21.0,28.4,...,40.5,13.0,17.6,10.0,13.5,23.0,31.1,0.838,1,83.8
4,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2019,All Students,27,606.0,1.0,3.7,...,25.9,18.0,66.7,1.0,3.7,19.0,70.4,0.845,1,84.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14462,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,All Grades,2019,All Students,323,593.0,113.0,35.0,...,35.3,73.0,22.6,23.0,7.1,96.0,29.7,0.949,1,94.9
14463,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,6,2018,All Students,111,595.0,37.0,33.3,...,33.3,26.0,23.4,11.0,9.9,37.0,33.3,0.915,1,91.5
14464,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,7,2018,All Students,116,592.0,50.0,43.1,...,41.4,18.0,15.5,0.0,0.0,18.0,15.5,0.915,1,91.5
14465,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,8,2018,All Students,90,591.0,24.0,26.7,...,53.3,16.0,17.8,2.0,2.2,18.0,20.0,0.915,1,91.5


In [17]:
# grade splits

data3 = grade_data[
    (grade_data['Grade'] == '3') & 
    (grade_data['Student Category'] == 'All Students') 
    # (grade_data['Report Category'] == 'School')
]


data4 = grade_data[
    (grade_data['Grade'] == '4') &  
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data5 = grade_data[
    (grade_data['Grade'] == '5') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data6 = grade_data[
    (grade_data['Grade'] == '6') &   
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data7 = grade_data[
    (grade_data['Grade'] == '7') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data8 = grade_data[
    (grade_data['Grade'] == '8') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]



In [18]:
# data3['poverty_2018'] = data3['poverty_percentage'] * (data3['Year'] == 2018).astype(int)
# data3['poverty_2022'] = data3['poverty_percentage'] * (data3['Year'] == 2022).astype(int)

In [19]:
def new_poverty_interaction(df):
    df['poverty_2018'] = df['poverty_percentage'] * (df['Year'] == 2018).astype(int)
    df['poverty_2022'] = df['poverty_percentage'] * (df['Year'] == 2022).astype(int)
    return df   

In [20]:
new_poverty_interaction(data3)
new_poverty_interaction(data4)
new_poverty_interaction(data5)
new_poverty_interaction(data6)
new_poverty_interaction(data7)
new_poverty_interaction(data8)

C:\Users\madis\AppData\Local\Temp\ipykernel_33604\3924902905.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['poverty_2018'] = df['poverty_percentage'] * (df['Year'] == 2018).astype(int)
C:\Users\madis\AppData\Local\Temp\ipykernel_33604\3924902905.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['poverty_2022'] = df['poverty_percentage'] * (df['Year'] == 2022).astype(int)
C:\Users\madis\AppData\Local\Temp\ipykernel_33604\3924902905.py:2: SettingWithCopyWarning: 
A value is trying to be set on a

,Report Category,DBN,school_name,Grade,Year,Student Category,number_tested,mean_scale_score,level_1_count,level_1_percentage,...,level_3_percentage,level_4_count,level_4_percentage,level_3_4_count,level_3_4_percentage,% Poverty,Poverty_Category,poverty_percentage,poverty_2018,poverty_2022
29,School,01M034,P.S. 034 FRANKLIN D. ROOSEVELT,8,2022,All Students,30,589.0,7.0,23.3,...,20.0,1.0,3.3,7.0,23.3,0.950,1,95.0,0.0,95.0
36,School,01M034,P.S. 034 FRANKLIN D. ROOSEVELT,8,2019,All Students,32,597.0,5.0,15.6,...,12.5,5.0,15.6,9.0,28.1,0.950,1,95.0,0.0,0.0
43,School,01M034,P.S. 034 FRANKLIN D. ROOSEVELT,8,2018,All Students,52,598.0,8.0,15.4,...,19.2,10.0,19.2,20.0,38.5,0.950,1,95.0,95.0,0.0
98,School,01M140,P.S. 140 NATHAN STRAUS,8,2022,All Students,45,596.0,7.0,15.6,...,26.7,5.0,11.1,17.0,37.8,0.865,1,86.5,0.0,86.5
105,School,01M140,P.S. 140 NATHAN STRAUS,8,2019,All Students,51,593.0,9.0,17.6,...,17.6,4.0,7.8,13.0,25.5,0.865,1,86.5,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14449,School,32K554,ALL CITY LEADERSHIP SECONDARY SCHOOL,8,2019,All Students,59,622.0,0.0,0.0,...,32.2,38.0,64.4,57.0,96.6,0.816,1,81.6,0.0,0.0
14453,School,32K554,ALL CITY LEADERSHIP SECONDARY SCHOOL,8,2018,All Students,59,622.0,0.0,0.0,...,32.2,40.0,67.8,59.0,100.0,0.857,1,85.7,85.7,0.0
14457,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,8,2022,All Students,83,595.0,20.0,24.1,...,24.1,13.0,15.7,33.0,39.8,0.925,1,92.5,0.0,92.5
14461,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,8,2019,All Students,109,596.0,16.0,14.7,...,25.7,8.0,7.3,36.0,33.0,0.949,1,94.9,0.0,0.0


In [21]:
data3 = data3.set_index(['DBN', 'Year'])
data4 = data4.set_index(['DBN', 'Year'])
data5 = data5.set_index(['DBN', 'Year'])
data6 = data6.set_index(['DBN', 'Year'])
data7 = data7.set_index(['DBN', 'Year'])
data8 = data8.set_index(['DBN', 'Year'])

In [22]:
# # verify index
# print(data3.index.names)    
# print(data3.shape)          

# year_counts = data3.reset_index().groupby("DBN")["Year"].count()
# # print(data3.reset_index().groupby("DBN")["Year"].count().value_counts())

# schools_to_keep = year_counts[year_counts >= 3].index

# # Filter the DataFrame to keep only those schools
# data3 = data3[data3.index.get_level_values('DBN').isin(schools_to_keep)]

# # Verify the result
# print(data3.reset_index().groupby("DBN")["Year"].count().value_counts())

In [23]:
def verify_index(df):

    ''' This function verifies that each school (DBN) has at least 3 years of data for the PanelOLS. Removes
    schools that do not include all three years.
    
    Input: 
        df: dataframe with indexed DBN and Year
    Output:
        df_filtered: filtered data that contains all schools for each grade with at all three years of data
    '''
    
    # count number of years in the data
    year_counts = df.reset_index().groupby("DBN")["Year"].count()
    # schools to keep are those with three years of data
    schools_to_keep = year_counts[year_counts >= 3].index
    # filter the data to keep only those schools
    
    df_filtered = df[df.index.get_level_values('DBN').isin(schools_to_keep)]
   
    return df_filtered

In [24]:
# apply the function to each grade dataset
data3 = verify_index(data3)
data4 = verify_index(data4) 
data5 = verify_index(data5)
data6 = verify_index(data6)
data7 = verify_index(data7)
data8 = verify_index(data8)

In [25]:
model3 = PanelOLS(
    dependent=data3['mean_scale_score'],
    exog=data3[['poverty_2018', 'poverty_2022']],
    entity_effects=True,
    time_effects=True,
    weights=data3['number_tested']
).fit(cov_type='clustered', cluster_entity=True)

print(model3.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.0258
Estimator:                   PanelOLS   R-squared (Between):             -0.0056
No. Observations:                2288   R-squared (Within):              -0.0759
Date:                Wed, Apr 29 2026   R-squared (Overall):             -0.0056
Time:                        13:02:04   Log-likelihood                   -5578.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      20.107
Entities:                         765   P-value                           0.0000
Avg Obs:                       2.9908   Distribution:                  F(2,1519)
Min Obs:                       2.0000                                           
Max Obs:                       3.0000   F-statistic (robust):             13.003
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [26]:
def run_grade_model(data, grade):
    model = PanelOLS(
        dependent=data['mean_scale_score'],
        exog=data[['poverty_2018', 'poverty_2022']],
        entity_effects=True,
        time_effects=True,
        weights=data['number_tested']
    ).fit(cov_type='clustered', cluster_entity=True)
    
    print(f"Grade {grade} → Coef 2018: {model.params['poverty_2018']:.4f}, "
          f"p-value 2018: {model.pvalues['poverty_2018']:.4f}; "
          f"Coef 2022: {model.params['poverty_2022']:.4f}, "
          f"p-value 2022: {model.pvalues['poverty_2022']:.4f}")
    
    return model

In [27]:
data3

Report Category                         school_name Grade  \
DBN    Year                                                             
01M015 2022          School           P.S. 015 ROBERTO CLEMENTE     3   
       2019          School           P.S. 015 ROBERTO CLEMENTE     3   
       2018          School           P.S. 015 ROBERTO CLEMENTE     3   
01M020 2022          School                P.S. 020 ANNA SILVER     3   
       2019          School                P.S. 020 ANNA SILVER     3   
...                     ...                                 ...   ...   
32K377 2019          School  P.S. 377 ALEJANDRINA B. DE GAUTIER     3   
       2018          School  P.S. 377 ALEJANDRINA B. DE GAUTIER     3   
32K384 2022          School    P.S. /I.S. 384 FRANCES E. CARTER     3   
       2019          School    P.S. /I.S. 384 FRANCES E. CARTER     3   
       2018          School    P.S. /I.S. 384 FRANCES E. CARTER     3   

            Student Category  number_tested  mean_scale_score  level_1_count  \
DBN    Year                                                                    
01M015 2022     All Students             21             594.0            4.0   
       2019     All Students             27             606.0            1.0   
       2018     All Students             20             613.0            0.0   
01M020 2022     All Students             21             593.0            6.0   
       2019     All Students             57             593.0           13.0   
...                      ...            ...               ...            ...   
32K377 2019     All Students             23             588.0            8.0   
       2018     All Students             16             586.0            3.0   
32K384 2022     All Students             38             584.0           13.0   
       2019     All Students             39             585.0           18.0   
       2018     All Students             41             592.0           13.0   

             level_1_percentage  level_2_count  level_2_percentage  ...  \
DBN    Year                                                         ...   
01M015 2022                19.0           12.0                57.1  ...   
       2019                 3.7            7.0                25.9  ...   
       2018                 0.0            4.0                20.0  ...   
01M020 2022                28.6            6.0                28.6  ...   
       2019                22.8           24.0                42.1  ...   
...                         ...            ...                 ...  ...   
32K377 2019                34.8           11.0                47.8  ...   
       2018                18.8           10.0                62.5  ...   
32K384 2022                34.2           19.0                50.0  ...   
       2019                46.2           12.0                30.8  ...   
       2018                31.7           13.0                31.7  ...   

             level_3_percentage  level_4_count  level_4_percentage  \
DBN    Year                                                          
01M015 2022                19.0            1.0                 4.8   
       2019                66.7            1.0                 3.7   
       2018                65.0            3.0                15.0   
01M020 2022                42.9            0.0                 0.0   
       2019                31.6            2.0                 3.5   
...                         ...            ...                 ...   
32K377 2019                17.4            0.0                 0.0   
       2018                18.8            0.0                 0.0   
32K384 2022                15.8            0.0                 0.0   
       2019                23.1            0.0                 0.0   
       2018                34.1            1.0                 2.4   

             level_3_4_count  level_3_4_percentage  % Poverty  \
DBN    Year                                                     
01M0

In [28]:
model3 = run_grade_model(data3, 3)


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Grade 3 → Coef 2018: -0.0245, p-value 2018: 0.0031; Coef 2022: -0.0511, p-value 2022: 0.0000


In [29]:
model3

Dep. Variable:,mean_scale_score,R-squared:,0.0258
Estimator:,PanelOLS,R-squared (Between):,-0.0056
No. Observations:,2288,R-squared (Within):,-0.0759
Date:,"Wed, Apr 29 2026",R-squared (Overall):,-0.0056
Time:,13:02:04,Log-likelihood,-5578.9
Cov. Estimator:,Clustered,,
,,F-statistic:,20.107
Entities:,765,P-value,0.0000
Avg Obs:,2.9908,Distribution:,"F(2,1519)"
Min Obs:,2.0000,,
Max Obs:,3.0000,F-statistic (robust):,13.003


In [30]:
model4 = run_grade_model(data4,4)

Grade 4 → Coef 2018: -0.0289, p-value 2018: 0.0009; Coef 2022: -0.0715, p-value 2022: 0.0000


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [31]:
model5 = run_grade_model(data5, 5)

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Grade 5 → Coef 2018: 0.0039, p-value 2018: 0.6184; Coef 2022: 0.0180, p-value 2022: 0.0517


In [32]:
model6 = run_grade_model(data6, 6)


Grade 6 → Coef 2018: 0.0304, p-value 2018: 0.0077; Coef 2022: 0.0867, p-value 2022: 0.0000


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [33]:
model7 = run_grade_model(data7, 7)


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Grade 7 → Coef 2018: 0.0296, p-value 2018: 0.0086; Coef 2022: 0.0769, p-value 2022: 0.0000


In [34]:
model8 = run_grade_model(data8, 8)

Grade 8 → Coef 2018: -0.0113, p-value 2018: 0.3250; Coef 2022: -0.0106, p-value 2022: 0.4696


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


#### Exports

In [35]:
import os

In [36]:
# os.makedirs("honestdid_coefs", exist_ok=True)

In [37]:
def export_coefficients(model, grade):
    ''' This function exports the coefficients and variance-covariance matrix of PanelOLS models to be imported
    into R for HonestDiD analysis.
    
    Input:
        model: PanelOLS model
        grade: grade level for naming the output files
    Output:
        two csv files containing coefficients and variance-covariance matrix for the model.
        '''
    
    np.savetxt(f"honestdid_coefs/beta_grade{grade}.csv",
               model.params.values, delimiter=",")
    np.savetxt(f"honestdid_coefs/vcov_grade{grade}.csv",
               model.cov.values, delimiter=",")

In [38]:
# # export for R
# export_coefficients(model3, 3)
# export_coefficients(model4, 4)
# export_coefficients(model5, 5)
# export_coefficients(model6, 6)
# export_coefficients(model7, 7)
# export_coefficients(model8, 8)

### Model 1: Pooled Ordinary Least Squares Model

Pooled OLS treats panel data as if it is one cross-sectional piece of data. Essentially, this is a baseline model where time and group fixed effects are ignored. <br>
https://www.geeksforgeeks.org/artificial-intelligence/pooled-ols-regression/

# Trying something

In [39]:
data3_test = data3.reset_index(['DBN', 'Year'])

In [40]:
data3_test

,DBN,Year,Report Category,school_name,Grade,Student Category,number_tested,mean_scale_score,level_1_count,level_1_percentage,...,level_3_percentage,level_4_count,level_4_percentage,level_3_4_count,level_3_4_percentage,% Poverty,Poverty_Category,poverty_percentage,poverty_2018,poverty_2022
0,01M015,2022,School,P.S. 015 ROBERTO CLEMENTE,3,All Students,21,594.0,4.0,19.0,...,19.0,1.0,4.8,5.0,23.8,0.838,1,83.8,0.0,83.8
1,01M015,2019,School,P.S. 015 ROBERTO CLEMENTE,3,All Students,27,606.0,1.0,3.7,...,66.7,1.0,3.7,19.0,70.4,0.845,1,84.5,0.0,0.0
2,01M015,2018,School,P.S. 015 ROBERTO CLEMENTE,3,All Students,20,613.0,0.0,0.0,...,65.0,3.0,15.0,16.0,80.0,0.847,1,84.7,84.7,0.0
3,01M020,2022,School,P.S. 020 ANNA SILVER,3,All Students,21,593.0,6.0,28.6,...,42.9,0.0,0.0,9.0,42.9,0.701,0,70.1,0.0,70.1
4,01M020,2019,School,P.S. 020 ANNA SILVER,3,All Students,57,593.0,13.0,22.8,...,31.6,2.0,3.5,20.0,35.1,0.678,0,67.8,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2293,32K377,2019,School,P.S. 377 ALEJANDRINA B. DE GAUTIER,3,All Students,23,588.0,8.0,34.8,...,17.4,0.0,0.0,4.0,17.4,0.929,1,92.9,0.0,0.0
2294,32K377,2018,School,P.S. 377 ALEJANDRINA B. DE GAUTIER,3,All Students,16,586.0,3.0,18.8,...,18.8,0.0,0.0,3.0,18.8,0.937,1,93.7,93.7,0.0
2295,32K384,2022,School,P.S. /I.S. 384 FRANCES E. CARTER,3,All Students,38,584.0,13.0,34.2,...,15.8,0.0,0.0,6.0,15.8,0.844,1,84.4,0.0,84.4
2296,32K384,2019,School,P.S. /I.S. 384 FRANCES E. CARTER,3,All Students,39,585.0,18.0,46.2,...,23.1,0.0,0.0,9.0,23.1,0.867,1,86.7,0.0,0.0


In [41]:
import pandas as pd
from linearmodels import PanelOLS

def run_grade_model(data):

    grade_data = data
    
    # 2019 poverty rate as baseline for each school
    poverty_baseline = grade_data[grade_data['Year'] == 2019][
        ['DBN', 'poverty_percentage']
    ].rename(columns={'poverty_percentage': 'poverty_baseline'})
    
    # merge the baseline poverty rate back into the main data
    grade_data = grade_data.merge(poverty_baseline, on='DBN', how='left')
    
    # interaction terms using new poverty baseline(2019) for pre and post periods
    grade_data['poverty_pre'] = (
        grade_data['poverty_baseline'] * (grade_data['Year'] == 2018).astype(int)
    )
    grade_data['poverty_post'] = (
        grade_data['poverty_baseline'] * (grade_data['Year'] == 2022).astype(int)
    )
    
    # index for PanelOLS
    grade_data = grade_data.set_index(['DBN', 'Year'])
    
    # PanelOLS model with clustered standard errors at the school level,
    # controlling for school and year fixed effects, 
    # and weighted by number of students tested
    model = PanelOLS(
        dependent=grade_data['mean_scale_score'],
        exog=grade_data[['poverty_pre', 'poverty_post']],
        entity_effects=True,
        time_effects=True,
        weights=grade_data['number_tested']
    ).fit(cov_type='clustered', cluster_entity=True)
    
    return model



model_3_test = run_grade_model(data3_test)


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [42]:
data4_test = data4.reset_index(['DBN', 'Year'])

model_4_test = run_grade_model(data4_test)

# model_4_test.summary

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [43]:
def min_max_scale_score(df):
    min_score = df['mean_scale_score'].min()
    max_score = df['mean_scale_score'].max()
    return min_score, max_score


In [44]:
grade_data

,Report Category,DBN,school_name,Grade,Year,Student Category,number_tested,mean_scale_score,level_1_count,level_1_percentage,...,level_2_percentage,level_3_count,level_3_percentage,level_4_count,level_4_percentage,level_3_4_count,level_3_4_percentage,% Poverty,Poverty_Category,poverty_percentage
0,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2022,All Students,21,594.0,4.0,19.0,...,57.1,4.0,19.0,1.0,4.8,5.0,23.8,0.838,1,83.8
1,School,01M015,P.S. 015 ROBERTO CLEMENTE,4,2022,All Students,30,596.0,6.0,20.0,...,46.7,5.0,16.7,5.0,16.7,10.0,33.3,0.838,1,83.8
2,School,01M015,P.S. 015 ROBERTO CLEMENTE,5,2022,All Students,23,599.0,11.0,47.8,...,17.4,4.0,17.4,4.0,17.4,8.0,34.8,0.838,1,83.8
3,School,01M015,P.S. 015 ROBERTO CLEMENTE,All Grades,2022,All Students,74,596.0,21.0,28.4,...,40.5,13.0,17.6,10.0,13.5,23.0,31.1,0.838,1,83.8
4,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2019,All Students,27,606.0,1.0,3.7,...,25.9,18.0,66.7,1.0,3.7,19.0,70.4,0.845,1,84.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14462,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,All Grades,2019,All Students,323,593.0,113.0,35.0,...,35.3,73.0,22.6,23.0,7.1,96.0,29.7,0.949,1,94.9
14463,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,6,2018,All Students,111,595.0,37.0,33.3,...,33.3,26.0,23.4,11.0,9.9,37.0,33.3,0.915,1,91.5
14464,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,7,2018,All Students,116,592.0,50.0,43.1,...,41.4,18.0,15.5,0.0,0.0,18.0,15.5,0.915,1,91.5
14465,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,8,2018,All Students,90,591.0,24.0,26.7,...,53.3,16.0,17.8,2.0,2.2,18.0,20.0,0.915,1,91.5


In [45]:
grade4_schools = set(grade_data[grade_data['Grade'] == '4']['DBN'])
grade5_schools = set(grade_data[grade_data['Grade'] == '5']['DBN'])
grade6_schools = set(grade_data[grade_data['Grade'] == '6']['DBN'])

print(f"Grade 4 only: {len(grade4_schools - grade5_schools)}")
print(f"Grade 5 only: {len(grade5_schools - grade4_schools)}")
print(f"Both 4 and 5: {len(grade4_schools & grade5_schools)}")

print(f"Grade 5 only: {len(grade5_schools - grade6_schools)}")
print(f"Grade 6 only: {len(grade6_schools - grade5_schools)}")
print(f"Both 5 and 6: {len(grade5_schools & grade6_schools)}")

Grade 4 only: 4
Grade 5 only: 5
Both 4 and 5: 775
Grade 5 only: 621
Grade 6 only: 331
Both 5 and 6: 159


In [46]:
grade_data

,Report Category,DBN,school_name,Grade,Year,Student Category,number_tested,mean_scale_score,level_1_count,level_1_percentage,...,level_2_percentage,level_3_count,level_3_percentage,level_4_count,level_4_percentage,level_3_4_count,level_3_4_percentage,% Poverty,Poverty_Category,poverty_percentage
0,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2022,All Students,21,594.0,4.0,19.0,...,57.1,4.0,19.0,1.0,4.8,5.0,23.8,0.838,1,83.8
1,School,01M015,P.S. 015 ROBERTO CLEMENTE,4,2022,All Students,30,596.0,6.0,20.0,...,46.7,5.0,16.7,5.0,16.7,10.0,33.3,0.838,1,83.8
2,School,01M015,P.S. 015 ROBERTO CLEMENTE,5,2022,All Students,23,599.0,11.0,47.8,...,17.4,4.0,17.4,4.0,17.4,8.0,34.8,0.838,1,83.8
3,School,01M015,P.S. 015 ROBERTO CLEMENTE,All Grades,2022,All Students,74,596.0,21.0,28.4,...,40.5,13.0,17.6,10.0,13.5,23.0,31.1,0.838,1,83.8
4,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2019,All Students,27,606.0,1.0,3.7,...,25.9,18.0,66.7,1.0,3.7,19.0,70.4,0.845,1,84.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14462,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,All Grades,2019,All Students,323,593.0,113.0,35.0,...,35.3,73.0,22.6,23.0,7.1,96.0,29.7,0.949,1,94.9
14463,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,6,2018,All Students,111,595.0,37.0,33.3,...,33.3,26.0,23.4,11.0,9.9,37.0,33.3,0.915,1,91.5
14464,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,7,2018,All Students,116,592.0,50.0,43.1,...,41.4,18.0,15.5,0.0,0.0,18.0,15.5,0.915,1,91.5
14465,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,8,2018,All Students,90,591.0,24.0,26.7,...,53.3,16.0,17.8,2.0,2.2,18.0,20.0,0.915,1,91.5


In [47]:

print(grade_data.groupby('Grade')['poverty_percentage'].describe())

             count       mean        std  min   25%    50%     75%   max
Grade                                                                   
3           2324.0  74.006411  22.642153  5.6  65.1  81.75  92.100  95.0
4           2296.0  73.997561  22.621523  5.6  65.0  81.75  92.100  95.0
5           2280.0  73.990219  22.640002  5.6  65.1  81.80  92.100  95.0
6           1435.0  76.623206  19.798401  8.4  69.7  83.10  91.700  95.0
7           1405.0  76.716584  19.796454  8.4  70.0  83.30  91.800  95.0
8           1402.0  76.780243  19.773282  8.4  70.0  83.35  91.875  95.0
All Grades  3325.0  75.452571  21.473371  5.6  67.6  82.90  92.100  95.0


In [48]:
data5_test = data5.reset_index(['DBN', 'Year'])
model_5_test = run_grade_model(data5_test)
# model_5_test.summary


data6_test = data6.reset_index(['DBN', 'Year'])
model_6_test = run_grade_model(data6_test)
# model_6_test.summary

data7_test = data7.reset_index(['DBN', 'Year'])
model_7_test = run_grade_model(data7_test)

data8_test = data8.reset_index(['DBN', 'Year'])
model_8_test = run_grade_model(data8_test)

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping 

In [49]:
os.makedirs("honestdid_coefs_test", exist_ok=True)

In [50]:
def export_coefficients2(model, grade):
    ''' This function exports the coefficients and variance-covariance matrix of PanelOLS models to be imported
    into R for HonestDiD analysis.
    
    Input:
        model: PanelOLS model
        grade: grade level for naming the output files
    Output:
        two csv files containing coefficients and variance-covariance matrix for the model.
        '''
    
    np.savetxt(f"honestdid_coefs_test/beta_grade{grade}.csv",
               model.params.values, delimiter=",")
    np.savetxt(f"honestdid_coefs_test/vcov_grade{grade}.csv",
               model.cov.values, delimiter=",")

In [51]:
# export_coefficients2(model_3_test, 3)
# export_coefficients2(model_4_test, 4)
# export_coefficients2(model_5_test, 5)
# export_coefficients2(model_6_test, 6)
# export_coefficients2(model_7_test, 7)
# export_coefficients2(model_8_test, 8)

In [59]:
import pandas as pd
from linearmodels import PanelOLS

def run_grade_model_levels(data):

    grade_data = data
    
    # 2019 poverty rate as baseline for each school
    poverty_baseline = grade_data[grade_data['Year'] == 2019][
        ['DBN', 'poverty_percentage']
    ].rename(columns={'poverty_percentage': 'poverty_baseline'})
    
    # merge the baseline poverty rate back into the main data
    grade_data = grade_data.merge(poverty_baseline, on='DBN', how='left')
    
    # interaction terms using new poverty baseline(2019) for pre and post periods
    grade_data['poverty_pre'] = (
        grade_data['poverty_baseline'] * (grade_data['Year'] == 2018).astype(int)
    )
    grade_data['poverty_post'] = (
        grade_data['poverty_baseline'] * (grade_data['Year'] == 2022).astype(int)
    )
    
    # index for PanelOLS
    grade_data = grade_data.set_index(['DBN', 'Year'])
    
    # PanelOLS model with clustered standard errors at the school level,
    # controlling for school and year fixed effects, 
    # and weighted by number of students tested
    model = PanelOLS(
        dependent=grade_data['level_4_percentage'],
        exog=grade_data[['poverty_pre', 'poverty_post']],
        entity_effects=True,
        time_effects=True,
        weights=grade_data['number_tested']
    ).fit(cov_type='clustered', cluster_entity=True)
    
    return model



model_3_level_test = run_grade_model_levels(data3_test)


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [60]:
model_3_level_test.summary

Dep. Variable:,level_4_percentage,R-squared:,0.0036
Estimator:,PanelOLS,R-squared (Between):,-0.0799
No. Observations:,2288,R-squared (Within):,-0.0180
Date:,"Wed, Apr 29 2026",R-squared (Overall):,-0.0753
Time:,13:03:31,Log-likelihood,-6243.9
Cov. Estimator:,Clustered,,
,,F-statistic:,2.7615
Entities:,765,P-value,0.0635
Avg Obs:,2.9908,Distribution:,"F(2,1519)"
Min Obs:,2.0000,,
Max Obs:,3.0000,F-statistic (robust):,0.9797


In [61]:
model_4_level_test = run_grade_model_levels(data4_test)

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [62]:
model_4_level_test.summary

Dep. Variable:,level_4_percentage,R-squared:,0.0217
Estimator:,PanelOLS,R-squared (Between):,-0.1715
No. Observations:,2244,R-squared (Within):,-0.3446
Date:,"Wed, Apr 29 2026",R-squared (Overall):,-0.1778
Time:,13:03:39,Log-likelihood,-6733.2
Cov. Estimator:,Clustered,,
,,F-statistic:,16.490
Entities:,751,P-value,0.0000
Avg Obs:,2.9880,Distribution:,"F(2,1489)"
Min Obs:,2.0000,,
Max Obs:,3.0000,F-statistic (robust):,8.4290
